### Дообучение с небольшим числом обучаемых параметров (PEFT)

В этом ноутбуке вы будете дообучать большие языковые модели при ограниченном объёме видеопамяти. Вы реализуете prompt tuning и LoRA, а также попробуете библиотеку Hugging Face PEFT.

Для выполнения заданий нужен GPU с поддержкой CUDA. Замените заглушки вида `<YOUR CODE HERE>` своим кодом; в исходном виде ячейки с такими заглушками не выполняются. Английские фразы, которые подаются модели, оставлены без перевода: они являются данными для упражнений. Названия API и переменных также сохранены.


In [ ]:
# Исходные версии библиотек
# %pip install --quiet transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 peft==0.5.0 bitsandbytes==0.41.2.post2

# Версии, рекомендованные для Colab в октябре 2025 года (спасибо Льву!)
%pip install --quiet "bitsandbytes==0.45.3" "transformers>=4.43,<4.46" "accelerate>=0.33,<0.36" "peft>=0.11.1" "optimum>=1.20.0" "sentencepiece"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "Для этой части задания нужен GPU с поддержкой CUDA"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 992.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.0/301.0 kB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.4/520.4 kB 21.3 MB/s eta

In [ ]:
model_name = 'Enoch/llama-7b-hf'

# Загружаем токенизатор Llama...
tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

# ...и саму модель
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # Веса — 4-битные; нормализации и активации — в fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # Сохраняем лишь часть активаций, остальные вычисляем повторно
model.enable_input_require_grads()     # Для checkpointing включаем вычисление градиентов по входам, чтобы сохранить обратное распространение
# Подробнее о gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:484: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_file, map_location=map

### Prompt tuning: история одной лисы (1 балл)

![Лиса из примера](https://i.imgur.com/Ux3qQAu.png) (источник: theodd1souts.fandom.com)


In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nРезультат:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: <s>A quick brown fox jumps over the lazy dog.
A quick


Какая наглая ложь! Эта лиса уверяет, что вовсе не перепрыгивала через ленивую собаку. Она просто занималась своими делами. **Ваша задача — обучить модель говорить правду: сегодня через собаку никто не прыгал.**

В примере используется начало известной английской панграммы: «A quick brown fox» — «Быстрая бурая лиса». Целевая фраза `the_truth` отрицает привычное продолжение «jumped over the lazy dog». Для обучения используйте именно английские строки из ячеек ниже.


In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Потери:", loss)

Loss: tensor(3.0725, device='cuda:0', grad_fn=<NllLossBackward0>)


Однако обучать всю модель мы не можем: только градиенты в формате `float32` заняли бы около 28 ГБ. Вместо этого применим [prompt tuning](https://arxiv.org/abs/2104.08691) — обучение небольшого набора векторов, добавляемых перед эмбеддингами входных токенов. Параметры исходной модели остаются замороженными.

![Схема prompt tuning](https://i.imgur.com/VwNNKnb.png)


In [ ]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    Для prompt tuning замените исходный слой эмбеддингов модели этим слоем.
    Он подставляет обучаемые векторы промпта вместо эмбеддингов первых num_prompts токенов.
    """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True
        )

    def forward(self, input_ids: torch.LongTensor):
        # Форма input_ids: [batch_size, seq_length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), (
            "Не забудьте добавить num_prompts PAD-токенов в начало input_ids"
        )

        # Получите эмбеддинги input_ids с помощью исходного слоя
        input_embeddings = <YOUR CODE HERE>  # Форма: [batch_size, seq_length, embedding_dim]

        # Замените эмбеддинги первых num_prompts токенов обучаемыми векторами промпта
        batch_size = input_ids.shape[0]
        learnable_prompts_expanded = <YOUR CODE HERE>  # Форма: [batch_size, num_prompts, embedding_dim]
        remaining_embeddings = <YOUR CODE HERE>  # Форма: [batch_size, seq_length - num_prompts, embedding_dim]

        # Объедините обучаемые векторы промпта с эмбеддингами оставшихся токенов
        output_embeddings = <YOUR CODE HERE>

        return output_embeddings


In [ ]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Проверки пройдены!")

Looks legit!


<ipython-input-6-acb6260f9ffb>:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


**Теперь, когда слой работает,** добавим обучаемые векторы промпта в основную модель и научим её рассказывать правду о лисе.


In [ ]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "Вы уже заменили слой эмбеддингов. Если новая реализация работает неверно, заново загрузите модель"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts : -1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Потери:", loss)


raise NotImplemented("Задание: обучите векторы промпта в цикле, уменьшая потери с помощью оптимизатора opt")

Epoch 1/100, Loss: 8.354496955871582
Epoch 2/100, Loss: 7.563575744628906
Epoch 3/100, Loss: 7.1476030349731445
Epoch 4/100, Loss: 6.815784931182861
Epoch 5/100, Loss: 6.516808032989502
Epoch 6/100, Loss: 6.2551469802856445
Epoch 7/100, Loss: 6.02204704284668
Epoch 8/100, Loss: 5.805267333984375
Epoch 9/100, Loss: 5.594377517700195
Epoch 10/100, Loss: 5.384930610656738
Epoch 11/100, Loss: 5.176473617553711
Epoch 12/100, Loss: 4.968852519989014
Epoch 13/100, Loss: 4.762547016143799
Epoch 14/100, Loss: 4.560352325439453
Epoch 15/100, Loss: 4.365894317626953
Epoch 16/100, Loss: 4.180633068084717
Epoch 17/100, Loss: 4.003462314605713
Epoch 18/100, Loss: 3.832592725753784
Epoch 19/100, Loss: 3.6667115688323975
Epoch 20/100, Loss: 3.5045828819274902
Epoch 21/100, Loss: 3.344529867172241
Epoch 22/100, Loss: 3.1853034496307373
Epoch 23/100, Loss: 3.0280065536499023
Epoch 24/100, Loss: 2.876392126083374
Epoch 25/100, Loss: 2.730309009552002
Epoch 26/100, Loss: 2.58361554145813
Epoch 27/100, Los

In [ ]:
# Итоговая проверка значения потерь
assert loss.item() <= 0.1
print("Отлично!")

Good job!


In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nРезультат:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# Если всё сделано верно, модель будет отрицать, что лиса перепрыгнула через ленивую собаку


Output: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Используем Hugging Face PEFT (2 балла)

[`peft`](https://huggingface.co/docs/peft/index) — библиотека для дообучения предобученных трансформеров с небольшим числом обучаемых параметров (**P**arameter **E**fficient **F**ine-**T**uning). Она предоставляет общий интерфейс для prompt tuning, prefix tuning и нескольких методов на основе адаптеров.

Перед выполнением следующей ячейки заново загрузите исходную модель из начала ноутбука: здесь нужен исходный слой эмбеддингов, без вашей обёртки `WordEmbeddingsWithLearnedPrompts`.


In [ ]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "Заново загрузите исходную модель"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # Для большинства методов PEFT этот вызов также изменяет исходный объект model
print("Обучаемых параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Всего параметров (без учёта квантования):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 3500478464


In [ ]:
# Задание: обучите модель с обёрткой PEFT так, чтобы потери при предсказании следующего токена стали меньше 0.1
# PAD-токены вручную добавлять больше не нужно, но первые num_virtual_tokens позиций logits нужно пропустить
# В конце сгенерируйте продолжение фразы и убедитесь, что модель усвоила правду

In [ ]:
# Задаём целевую фразу
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

In [ ]:
# Настройки обучения
loss_threshold = 0.1  # Целевой порог потерь
num_epochs = 100  # Максимальное число эпох
learning_rate = <YOUR CODE HERE>  # Скорость обучения

# Создайте оптимизатор для обучаемых параметров — векторов промпта PEFT
optimizer = <YOUR CODE HERE>

# Задаём целевую фразу
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

# Цикл обучения
for epoch in range(num_epochs):
    # Прямой проход
    outputs = <YOUR CODE HERE>

    # Пропустите logits для виртуальных токенов и последнего токена
    next_word_logits = <YOUR CODE HERE>  # Пропускаем виртуальные токены
    true_next_tokens = <YOUR CODE HERE>  # Сдвигаем целевые токены на одну позицию

    # Вычисляем потери
    loss = F.cross_entropy(
        <Flatten logits>,
        <Flatten ground truth tokens>
    )

    # Обратное распространение и обновление параметров
    <YOUR CODE HERE>

    # Выводим потери, чтобы отслеживать обучение
    print(f"Эпоха {epoch + 1}/{num_epochs}, потери: {loss.item()}")

    # Останавливаем обучение, если потери ниже порога
    if loss.item() < loss_threshold:
        print("Целевой порог потерь достигнут. Останавливаем обучение.")
        break
else:
    print("Достигнуто максимальное число эпох; целевой порог потерь ещё не достигнут.")

Epoch 1/100, Loss: 7.675178050994873
Epoch 2/100, Loss: 6.892231464385986
Epoch 3/100, Loss: 6.3485493659973145
Epoch 4/100, Loss: 5.904366970062256
Epoch 5/100, Loss: 5.507501125335693
Epoch 6/100, Loss: 5.568404197692871
Epoch 7/100, Loss: 5.064216136932373
Epoch 8/100, Loss: 4.949523448944092
Epoch 9/100, Loss: 4.826657772064209
Epoch 10/100, Loss: 4.69673490524292
Epoch 11/100, Loss: 4.5657548904418945
Epoch 12/100, Loss: 4.434375286102295
Epoch 13/100, Loss: 4.301575660705566
Epoch 14/100, Loss: 4.167079448699951
Epoch 15/100, Loss: 4.031871318817139
Epoch 16/100, Loss: 3.897719383239746
Epoch 17/100, Loss: 3.766268730163574
Epoch 18/100, Loss: 3.6382246017456055
Epoch 19/100, Loss: 3.512860059738159
Epoch 20/100, Loss: 3.387976884841919
Epoch 21/100, Loss: 3.2605791091918945
Epoch 22/100, Loss: 3.128392219543457
Epoch 23/100, Loss: 2.991361141204834
Epoch 24/100, Loss: 2.8520336151123047
Epoch 25/100, Loss: 2.7146761417388916
Epoch 26/100, Loss: 2.5834615230560303
Epoch 27/100, L

In [ ]:
# Итоговая проверка: потери должны быть ниже порога
assert loss.item() < loss_threshold, "Не удалось снизить потери ниже заданного порога."
print("Обучение завершено успешно! Потери ниже 0.1.")

Training successful! Loss is below 0.1.


In [ ]:
prompt = "A quick brown fox"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

# Генерируем 15 токенов
for i in range(15):
    # Прямой проход для получения logits
    outputs = model(**batch)
    next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

    # Добавляем следующий токен к input_ids
    batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)

    # Дополняем attention_mask, чтобы её длина совпадала с новой длиной input_ids
    new_attention_mask = torch.ones_like(next_token, dtype=batch["attention_mask"].dtype).to(device)
    batch["attention_mask"] = torch.cat([batch["attention_mask"], new_attention_mask], dim=-1)

# Декодируем сгенерированную последовательность
# Здесь виртуальные токены добавляет PEFT внутри модели; в batch["input_ids"] их нет
decoded_output = tokenizer.decode(batch["input_ids"][0].cpu().numpy().tolist(), skip_special_tokens=True)
print("\nРезультат:", decoded_output)



Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Дообучение с помощью LoRA (2 балла)

Для более содержательных задач можно использовать адаптеры низкого ранга из [статьи о LoRA](https://arxiv.org/pdf/2106.09685.pdf).

Основная идея — добавить обучаемый адаптер низкого ранга **параллельно существующему линейному слою** и сложить их выходы:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

В исходной статье LoRA адаптеры добавляли только к матрицам проекций механизма внимания. Однако [последующие работы](https://arxiv.org/abs/2305.14314) показывают, что полезно адаптировать и полносвязные блоки (FFN). Прежде чем обучать модель, реализуйте базовый слой LoRA.


In [ ]:
# Заново загружаем модель без ранее добавленных адаптеров PEFT
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # Веса — 4-битные; нормализации и активации — в fp32
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
class LoRALayer(nn.Module):
    """Добавляет LoRA-адаптер к существующему линейному слою."""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # Предобученный замороженный линейный слой
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Примените self.module и LoRA-адаптер к входу, затем верните сумму их выходов
        original_output = <YOUR CODE HERE>
        lora_output = <YOUR CODE HERE>

        return original_output + lora_output

In [ ]:
# Проверяем вашу реализацию
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "Проверьте реализацию прямого прохода"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "Для некоторых параметров адаптера не вычислен градиент"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "Неверный градиент по матрице A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "Неверный градиент по матрице B"
# Неверный градиент: реализация отличается от формулы LoRA или нарушено вычисление градиентов (например, используется no_grad)
del dummy_loss, test_linear, test_adapter
print("Все проверки пройдены!")

All tests passed!


### Добавляем LoRA в модель

Код ниже добавляет LoRA-адаптеры к линейным проекциям Q/K/V в механизме внимания Llama. Можно также модифицировать другие слои:

* `self_attn.o_proj` — выходную проекцию механизма внимания;
* `mlp.up_proj`, `mlp.gate_proj`, `mlp.down_proj` — полносвязные слои трансформера;
* `lm_head` — выходной слой языковой модели.

Проверка в следующей ячейке ожидает 96 адаптеров: по три проекции Q/K/V в каждом из 32 блоков Llama-7B. Дополнительные адаптеры для итогового задания добавляйте после этой проверки.


In [ ]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # Для Llama-7B

In [ ]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False)
# Проверяем один шаг обучения: градиенты должны быть ненулевыми
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Проверка градиентов пройдена, отлично!")

Grad check successful, well done!


### Пример обучения модели

Пример ниже показывает, как обучать LoRA-адаптеры на небольшом демонстрационном датасете. В итоговом задании вам предстоит организовать похожее обучение.

**Обратите внимание:** условия итогового задания находятся ниже. Эта ячейка — только пример.


In [ ]:
# Проверяем, способна ли модель обучаться. Для полноценного обучения измените max_steps
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 примера
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # Отключаем предупреждение Trainer из Hugging Face

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=1, # Накопление градиентов увеличивает эффективный размер батча
        warmup_steps=250, 
        max_steps=100, 
        learning_rate=2e-4, 
        fp16=True,
        logging_steps=1, 
        output_dir='outputs', 
        report_to=None,
        save_strategy="no" # Для совместимости по состоянию на октябрь 2025 года (спасибо Владу!) 
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# При предупреждениях о кэше задайте model.config.use_cache = False. Перед генерацией включите кэш обратно!

trainer.train()

# Это только пример: можно прервать его, не дожидаясь завершения обучения :)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
1,1.891200
2,1.696000
3,0.896900
4,1.744600
5,1.168000
6,0.730000
7,1.525700
8,1.063700
9,0.669600
10,1.427400


TrainOutput(global_step=100, training_loss=0.5410390722751618, metrics={'train_runtime': 150.7473, 'train_samples_per_second': 1.327, 'train_steps_per_second': 0.663, 'total_flos': 621258424123392.0, 'train_loss': 0.5410390722751618, 'epoch': 6.25})

### Итоговое задание: обучаем модель на практике (5 баллов)

Ваша задача — дообучить модель **генерировать код на Python**. Используйте предыдущие примеры как основу.

* **Данные:** используйте [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) или другой датасет с кодом на Python. Для этого упражнения достаточно небольшой части обучающей выборки.
* **Предобработка:** оставьте файлы с расширением `.py`. Для `codeparrot-clean` этот шаг можно пропустить: он уже содержит код на Python.
* **Длина примеров:** возьмите первые 512 символов каждого примера. Здесь ограничение задано в символах, а не в токенах.
* **Адаптеры:** используйте реализованную выше LoRA **и хотя бы одно** из следующих дополнений:
  - адаптер для `lm_head`;
  - адаптеры для компонентов MLP (`mlp.*`);
  - обучаемые входные эмбеддинги (потребуется скорректировать использование памяти).
* **Обучение:** обучать модель до сходимости не обязательно. Ориентир — 500 шагов: после этого проверьте, генерирует ли модель код методом `.generate()`. Используйте эффективный размер батча не меньше 4 и накопление градиентов `gradient_accumulation_steps=4`. Эффективный размер батча равен `per_device_train_batch_size × gradient_accumulation_steps × число GPU`.

В библиотеке `peft` тоже есть реализация LoRA. Однако для этого задания нужно показать **хотя бы один полный запуск обучения с вашей собственной реализацией LoRA**.

**Альтернативное задание:** вместо генерации кода на Python можно выбрать другой датасет и задачу, например тексты любимого исполнителя или расшифровки подкаста, если такое использование данных этично. Покажите на примерах, чему модель научилась — или не научилась, — сравнив генерации до и после дообучения, как в шаблоне ниже.


In [ ]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # Можно добавить другие промпты, в том числе не связанные напрямую с Python

# <A WHOLE LOT OF YOUR CODE>
# До дообучения сгенерируйте исходные примеры для выбранных промптов
# Используйте transformers.Trainer, как в примере выше, или собственный код обучения
# После обучения покажите генерации модели: ожидаются фрагменты кода на Python
# Для удобства сравнения оформите генерации в таблицу, например с помощью pandas или HTML
# Модель с LoRA генерирует текст тем же способом, что и исходная модель в примерах выше

In [ ]:
# Этот шаблон позволяет сравнить сгенерированные примеры кода в таблице
# Можно оформить результаты другим удобным способом

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">ПРОМПТ</th>
    <th style="text-align: center; border:1px solid black">ДО ОБУЧЕНИЯ</th>
    <th style="text-align: center; border:1px solid black">ПОСЛЕ ОБУЧЕНИЯ</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []

for prompt in prompts:
    # Замените строки-заглушки в аргументах format() реальными генерациями
    rows.append(row_template.format(prompt, "ГЕНЕРАЦИЯ ДО ДООБУЧЕНИЯ", "ГЕНЕРАЦИЯ ПОСЛЕ ДООБУЧЕНИЯ"))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`import`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`from`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`while`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`try`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`if`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`for`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING
`torch`,BEFORE FINETUNING,TO BE GENERATED AFTER FINETUNING


Если вы дошли до этого места — поздравляем! Вы выполнили все основные задания.

**Дополнительное задание (бонус):** попробуйте реализовать обучение префиксов, передаваемых в механизм внимания через `model.forward(..., past_key_values=your_prompts)`. О вариантах обучения промптов и префиксов можно прочитать в статьях [1](https://arxiv.org/abs/2104.08691) и [2](https://arxiv.org/abs/2101.00190).

### Дополнительные материалы

* [Квантование после обучения](https://arxiv.org/abs/2208.07339)
* [Запуск больших моделей с помощью Accelerate](https://huggingface.co/docs/accelerate/package_reference/big_modeling)
* [Библиотека адаптеров AdapterHub](https://adapterhub.ml/)

### Запуск других моделей

Этот подход можно адаптировать к другим моделям сопоставимого размера, например [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) или [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). Потребуются небольшие изменения кода:

1. Измените имя модели в `AutoModelForCausalLM.from_pretrained()` и загрузите подходящий токенизатор через `AutoTokenizer`.
2. В коде prompt tuning замените `model.model.embed_tokens` на путь к слою эмбеддингов выбранной модели. Найти его поможет `print(model)`.
3. Адаптируйте код добавления LoRA-слоёв: названия блоков трансформера и их компонентов у других моделей отличаются.
